In [1]:
import os, torch

os.environ["KERAS_BACKEND"] = "torch"  # Keras 백엔드 설정

# GPU 잘 잡혔는지 확인용 (RTX 5060 Ti 출력되면 성공)
print(
    "🚀 사용 중인 GPU:",
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
)

🚀 사용 중인 GPU: NVIDIA GeForce RTX 5060 Ti


---
# RNN (Recurrent Neural Network, 순환 신경망)

## 1. 왜 RNN이 필요한가?
MLP·CNN은 **입력을 한 번에 통째로** 받는다. 그런데 세상에는 **순서가 의미를 만드는 데이터**가 있다.

| 데이터 종류 | 예시 | 순서가 바뀌면? |
|---|---|---|
| 자연어 | "나는 밥을 먹었다" | "밥을 나는 먹었다" → 어색/의미 변화 |
| 시계열 | 주가, 기온, 판매량 | 완전히 다른 흐름이 됨 |
| 음성/음악 | 파형, 멜로디 | 다른 소리가 됨 |

이런 데이터를 **시퀀스 데이터(Sequence Data)** 라고 하고, 순서 정보를 처리하려면
**"앞에서 본 내용을 기억한 채 다음 입력을 보는"** 구조가 필요하다 → RNN

## 2. RNN의 핵심 구조
```
        h0(0으로 시작)
         │
  x1 ──▶ [RNN] ──▶ h1
                    │
             x2 ──▶ [RNN] ──▶ h2
                              │
                       x3 ──▶ [RNN] ──▶ h3
                                        │
                                 x4 ──▶ [RNN] ──▶ h4 ──▶ [Dense] ──▶ 예측
```
- **은닉상태(hidden state, h)** : 지금까지 본 내용을 요약해 담은 **기억 저장소**
- 매 시점(Time Step)마다 `새로운 h = f(이번 입력 x + 이전 기억 h)`
- ★ **위 그림의 [RNN]은 서로 다른 층이 아니라 "같은 층을 계속 재사용"** 하는 것
  - 그래서 이름이 **Recurrent(순환)** → CNN의 가중치 공유와 같은 아이디어
- 활성화 함수는 기본이 **tanh** (-1 ~ 1) : 값이 계속 누적되므로 범위를 잡아줘야 함

## 3. 입력 데이터의 형태 (가장 헷갈리는 부분!)
RNN 입력은 반드시 **3차원**이어야 한다.

```
(samples, timesteps, features)
   샘플수    순환횟수    특성개수
```

| 축 | 의미 | 이번 실습에서 |
|---|---|---|
| samples | 데이터가 몇 개인가 | 5 (단어 5개) |
| timesteps | 몇 번에 나눠서 볼 것인가 | 4 (앞 글자 4개) |
| features | 한 시점에 들어가는 숫자 개수 | 14 (알파벳 14종 원핫) |

> 비교) 이미지 CNN 입력 : `(samples, height, width, channels)` = (2000, 224, 224, 3)

## 4. RNN의 한계 (→ LSTM / GRU 로 발전)
- 시퀀스가 길어지면 앞쪽 정보가 뒤로 갈수록 희미해짐 = **장기 의존성 문제**
- 역전파 시 같은 가중치를 계속 곱하므로 **기울기 소실/폭주(Vanishing/Exploding Gradient)**
- 해결책 : **LSTM**(기억 셀 + 3개 게이트), **GRU**(LSTM 간소화, 2개 게이트)

## 5. 오늘의 실습 목표
- 앞의 **4글자**를 보고 **다음에 올 1글자**를 맞추는 SimpleRNN 모델 만들기
- 예) `h, e, l, l` → `o`
---

### 간단하게 RNN 체험 (알파벳단위로 학습)
- SimpleRNN 맛보기
- 과거의 알파벳을 기억하여 다음 등장할 알파벳을 예측하는 모델 만들기(앞의 4글자가 들어가면 다음에 올 글자를 맞추는 모델을 만들기)
- hello, apple, happy, drink, house
- 단어사전(14개) : h,e,l,o,a,p,y,d,r,i,n,k,u,s
- 여러가지 방법 중 간단하게 원핫인코딩으로 수치화 진행

In [2]:
# 각각의 알파벳을 원핫인코딩으로 정의
#  - 컴퓨터는 글자를 모르므로 반드시 '숫자'로 바꿔줘야 함(수치화)
#  - h=0, e=1, l=2 ... 처럼 그냥 번호를 매기면 컴퓨터가 "l(2)이 h(0)보다 크다",
#    "h + e = l" 같은 엉뚱한 크기/순서 관계를 학습해버림
#  - 그래서 각 글자를 '자기 자리만 1인 벡터'로 표현 -> 모든 글자가 서로 동등해짐
#  - 단어사전 14개(h,e,l,o,a,p,y,d,r,i,n,k,u,s) 이므로 벡터 길이도 14

#      index :  0 1 2 3 4 5 6 7 8 9 10 11 12 13
#               h e l o a p y d r i  n  k  u  s
h = [1,0,0,0,0,0,0,0,0,0,0,0,0,0]   # index 0
e = [0,1,0,0,0,0,0,0,0,0,0,0,0,0]   # index 1
l = [0,0,1,0,0,0,0,0,0,0,0,0,0,0]   # index 2
o = [0,0,0,1,0,0,0,0,0,0,0,0,0,0]   # index 3
a = [0,0,0,0,1,0,0,0,0,0,0,0,0,0]   # index 4
p = [0,0,0,0,0,1,0,0,0,0,0,0,0,0]   # index 5
y = [0,0,0,0,0,0,1,0,0,0,0,0,0,0]   # index 6
d = [0,0,0,0,0,0,0,1,0,0,0,0,0,0]   # index 7
r = [0,0,0,0,0,0,0,0,1,0,0,0,0,0]   # index 8
i = [0,0,0,0,0,0,0,0,0,1,0,0,0,0]   # index 9
n = [0,0,0,0,0,0,0,0,0,0,1,0,0,0]   # index 10
k = [0,0,0,0,0,0,0,0,0,0,0,1,0,0]   # index 11
u = [0,0,0,0,0,0,0,0,0,0,0,0,1,0]   # index 12
s = [0,0,0,0,0,0,0,0,0,0,0,0,0,1]   # index 13

# ※ 실무에서는 글자/단어 종류가 수만 개라 원핫이 너무 커짐
#    -> Embedding 층으로 짧은 실수 벡터로 압축해서 사용 (다음에 배울 내용)

In [3]:
# 문제 데이터 정의
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# X : 각 단어의 '앞 4글자' -> 이것이 시퀀스(순서대로 들어갈 입력)
#     hello  -> h,e,l,l
#     apple  -> a,p,p,l
#     happy  -> h,a,p,p
#     drink  -> d,r,i,n
#     house  -> h,o,u,s
X = np.array([
    [h,e,l,l],
    [a,p,p,l],
    [h,a,p,p],
    [d,r,i,n],
    [h,o,u,s]
])

# 정답데이터 : 각 단어의 '마지막 1글자' (원핫 상태 그대로 -> 다중분류 정답 형태)
#  ※ 여기가 RNN의 핵심 포인트!
#     첫 글자가 h로 같은 hello / happy / house 인데 정답은 각각 o / y / e 로 다름
#     -> 첫 글자만 봐서는 절대 못 맞추고, '앞의 4글자 순서 전체'를 기억해야 맞출 수 있음
y1 = np.array([
    o,   # hell -> o
    e,   # appl -> e
    y,   # happ -> y
    k,   # drin -> k
    e    # hous -> e
])

In [4]:
# 데이터 크기 확인
X.shape, y1.shape

# X -> (5, 4, 14)
#      (샘플수, 순환횟수(Time Step), 특성개수)
#       5개단어   4글자씩 순환         알파벳 14종 원핫
#      ★ RNN 입력은 무조건 3차원! (2차원이면 reshape 필요)

# y1 -> (5, 14)
#       (샘플수, 클래스 개수) -> 마지막 1글자를 14개 중 하나로 분류하는 문제

# 비교) 이미지 데이터
# (2000, 224, 224, 3)
# (샘플수, 높이, 너비, 색상정보)

((5, 4, 14), (5, 14))

### RNN 구조로 신경망 쌓기

#### SimpleRNN 층의 주요 옵션
| 옵션 | 의미 |
|---|---|
| `units` | 은닉상태(기억)의 크기 = 뉴런 개수. **클수록 더 많은 정보를 기억** |
| `activation` | 기본값 **`tanh`** (보통 그대로 사용). 값이 순환하며 누적되므로 -1~1로 눌러줘야 발산하지 않음 |
| `return_sequences` | `False`(기본) : **마지막 시점의 결과 1개만** 출력 → 뒤에 Dense를 붙일 때<br>`True` : **모든 시점의 결과**를 출력 → RNN 층을 **여러 개 쌓을 때 반드시 True** |

#### 파라미터 개수 계산식
```
SimpleRNN 파라미터 = (입력특성 + units) × units + units(bias)
예) (14 + 3) × 3 + 3 = 54개
      ↑입력가중치   ↑순환가중치(이전 기억을 받는 부분)
```
> ★ **timesteps(4)가 계산식에 없다는 것이 핵심!**
> 시퀀스가 4든 100이든 같은 가중치를 재사용하므로 파라미터 수는 변하지 않음 (가중치 공유)

In [5]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import InputLayer, Dense, SimpleRNN

# ※ 에러 메모 : Adam 을 layers 에서 import 하려다 ImportError 발생했었음
#    Adam은 '층(layer)'이 아니라 '최적화 함수(optimizer)' -> 아래 경로가 맞음
#    from tensorflow.keras.optimizers import Adam

In [6]:
# 샘플 1개의 형태 확인 -> (4, 14) = (timesteps, features)
# InputLayer의 shape에는 '샘플 1개의 형태'만 적으면 됨 (샘플수는 제외)
X[0].shape

(4, 14)

In [7]:
# 1. 신경망 구조 설계
# 뼈대 생성
model = Sequential()

# 입력층
#  (4, 14) = (timesteps, features) -> 14개짜리 벡터를 4번에 나눠서 넣겠다는 뜻
model.add(InputLayer(shape=(4,14)))

# 중간층
#  SimpleRNN(units=3) : 기억(은닉상태)의 크기를 3으로 설정
#   - RNN 구조에서는 활성화 함수로 tanh 사용 (기본값이라 생략 가능)
#   - return_sequences 기본값 False -> 4번 순환한 뒤 '마지막 기억 1개'만 출력
#   - 출력 shape : (None, 3)   / 파라미터 : (14 + 3) * 3 + 3 = 54개
model.add(SimpleRNN(units=3))

# 출력층
#  14개 알파벳 중 하나를 고르는 다중분류 -> units=14 + softmax
model.add(Dense(units=14, activation='softmax'))

model.summary()

# 2. 학습 방법 및 평가 방법 설정
#  정답(y1)이 원핫인코딩 형태 -> categorical_crossentropy
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# 3. 학습
#  데이터가 5개뿐이라 검증 데이터를 나누지 않고 전부 학습에 사용
#  (일반화 성능을 보려는 게 아니라 'RNN이 순서를 기억하는지' 확인하는 체험용 실습)
#  샘플이 적어 epochs를 200으로 크게 잡아야 정확도 1.0에 도달함
model.fit(
    X, y1,
    epochs=200
)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 3)              │            54 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 14)             │            56 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 110 (440.00 B)

 Trainable params: 110 (440.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 794ms/step - accuracy: 0.0000e+00 - loss: 2.5931
Epoch 2/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.0000e+00 - loss: 2.5875
Epoch 3/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.0000e+00 - loss: 2.5820
Epoch 4/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0000e+00 - loss: 2.5764
Epoch 5/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.0000e+00 - loss: 2.5709
Epoch 6/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.2000 - loss: 2.5654
Epoch 7/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.2000 - loss: 2.5599
Epoch 8/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.2000 - loss: 2.5543
Epoch 9/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.2000 - loss: 2.5488
Epoch 10/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.2000 - loss: 2.5433
Epoch 11/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.2000 - loss: 2.5378
Epoch 12/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - 

In [8]:
# 모델 예측
#  ★ reshape(1, 4, 14) 를 하는 이유
#     X[0]의 shape은 (4, 14) 인데, 모델은 항상 '여러 개의 샘플 묶음(배치)'을 기대함
#     -> 맨 앞에 샘플 수 1을 붙여 3차원으로 만들어줘야 함
#
#  ★ argmax() 를 하는 이유
#     predict 결과는 softmax를 거친 14개의 '확률'
#     -> 그중 가장 확률이 높은 자리의 '번호(index)'를 뽑아야 어떤 글자인지 알 수 있음
model.predict(X[0].reshape(1,4,14)).argmax()

# 결과 3 -> 단어사전 index 3 = 'o'
# 입력이 h,e,l,l 이었으므로 hello 의 'o' 를 정확히 예측한 것!

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step


np.int64(1)

---
## 정리 : MLP / CNN / RNN 비교

| | MLP | CNN | RNN |
|---|---|---|---|
| 주 사용처 | 정형 데이터 | 이미지 | 시퀀스(텍스트/시계열/음성) |
| 입력 형태 | `(samples, features)` | `(samples, H, W, C)` | `(samples, timesteps, features)` |
| 핵심 층 | `Dense` | `Conv2D` + `MaxPooling2D` | `SimpleRNN` / `LSTM` / `GRU` |
| 가중치 공유 | ❌ | ⭕ (필터를 이미지 전체에 재사용) | ⭕ (같은 층을 시점마다 재사용) |
| 주요 활성화 | relu | relu | **tanh** |
| 잡아내는 것 | 특성 간 조합 | **공간적** 패턴 | **시간적/순서적** 패턴 |

## 오늘 얻은 감각
- 딥러닝 모델 설계는 결국 **"데이터의 구조에 맞는 층을 고르는 일"**
  - 위치 관계가 중요 → CNN / 순서가 중요 → RNN
- 세 모델 모두 마지막은 **Dense + softmax 분류부**로 끝난다는 공통점
- CNN의 '가중치 공유' 아이디어가 RNN에서도 똑같이 반복됨

## 다음에 이어질 내용
- `return_sequences=True` 로 RNN 층 여러 개 쌓기
- **LSTM / GRU** : 기울기 소실과 장기 의존성 문제 해결
- **Embedding** : 원핫(14차원, 대부분 0)을 짧고 의미 있는 실수 벡터로 압축
---